<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Decision Trees</h1></center>

Decision Trees recursively partition the feature space by choosing the split that most reduces impurity at each node.

### Topics:
1. [Theory — CART & Impurity Measures](#1)
2. [Dataset & Visualization](#2)
3. [Depth vs Accuracy](#3)
4. [Visualising the Tree](#4)
5. [Pruning & Feature Importance](#5)

In [ ]:
import seaborn as sns
dt_colors = ['#7b2d00', '#c1440e', '#e8b298', '#f4d9cc', '#3d1700']
print("Decision Tree Colors:")
sns.palplot(sns.color_palette(dt_colors))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
np.random.seed(42)
print("Libraries loaded.")

## Let's Start!

<a id = "1"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Theory — CART & Impurity Measures</h1></center>

## How CART Works

The **Classification And Regression Tree (CART)** algorithm:
1. For each feature and each possible split value, compute the weighted impurity of both child nodes
2. Choose the split that gives the **largest impurity reduction**
3. Recurse until stopping criterion is met (max_depth, min_samples, etc.)

## Gini Impurity

$$G = 1 - \sum_{k=1}^{K} p_k^2$$

Ranges from 0 (pure node) to 0.5 (maximally mixed for binary case).

## Entropy

$$H = -\sum_{k=1}^{K} p_k \log_2 p_k$$

Information Gain = parent entropy − weighted child entropy.

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'], ['\\(','\\)']]}});</SCRIPT>

In [ ]:
# Gini vs Entropy visualisation
p = np.linspace(0.001, 0.999, 300)
gini    = 1 - p**2 - (1-p)**2
entropy = -(p*np.log2(p) + (1-p)*np.log2(1-p))

plt.figure(figsize=(8, 4), dpi=100)
plt.plot(p, gini,        color="#c1440e", lw=2.5, label="Gini")
plt.plot(p, entropy / 2, color="#7b2d00", lw=2.5, ls="--", label="Entropy / 2")
plt.axvline(0.5, color="gray", lw=0.8, ls=":")
plt.xlabel("p (fraction positive class)")
plt.ylabel("Impurity")
plt.title("Gini vs Entropy — both peak at p=0.5")
plt.legend()
plt.show()

<a id = "2"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Breast Cancer Dataset</h1></center>

<center><h1 style = "background:#7b2d00 ;color:white;border:0;font-weight:bold">Information About Dataset</h1></center>

**Breast Cancer Wisconsin** — 569 samples, 30 features, binary target (malignant/benign).

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
df = X.copy()
df["target"] = y
print(f"Shape: {df.shape}")
print(f"Classes: {data.target_names}")
print(f"Class distribution:\n{pd.Series(y).value_counts()}")
df.describe().T.head(10)

<center><h1 style = "background:#c1440e ;color:white;border:0;font-weight:bold">Data Visualization</h1></center>

In [ ]:
top_features = ["mean radius", "mean texture", "mean area", "mean concavity"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=100)

for ax, feat in zip(axes.ravel(), top_features):
    for label, color in [(0, "#7b2d00"), (1, "#e8b298")]:
        sns.kdeplot(df[df["target"]==label][feat], ax=ax,
                    label=data.target_names[label], color=color, fill=True, alpha=0.4)
    ax.set_title(feat)
    ax.legend()

plt.suptitle("Feature Distributions by Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8), dpi=100)
sns.heatmap(X.iloc[:, :10].assign(target=y).corr(numeric_only=True),
            annot=True, fmt=".2f", cmap="Oranges", linewidths=0.5)
plt.title("Correlation Heatmap (first 10 features)")
plt.show()

<center><h1 style = "background:#e8b298 ;color:black;border:0;font-weight:bold">Train-Test Split</h1></center>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Positive rate — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")

<a id = "3"></a>
<center><h1 style = "background:#f4d9cc ;color:black;border:0;font-weight:bold">Depth vs Accuracy</h1></center>

In [ ]:
depths = range(1, 21)
train_acc, test_acc, cv_acc = [], [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, dt.predict(X_train)))
    test_acc.append(accuracy_score(y_test,  dt.predict(X_test)))
    cv = cross_val_score(dt, X_train, y_train, cv=5, scoring="accuracy")
    cv_acc.append(cv.mean())

best_depth = list(depths)[np.argmax(cv_acc)]
print(f"Best depth by 5-fold CV: {best_depth}  (CV acc = {max(cv_acc):.4f})")

plt.figure(figsize=(10, 5), dpi=100)
plt.plot(list(depths), train_acc, "o-", label="Train accuracy", color="#7b2d00")
plt.plot(list(depths), test_acc,  "s-", label="Test accuracy",  color="#c1440e")
plt.plot(list(depths), cv_acc,    "d--", label="CV accuracy",   color="#e8b298")
plt.axvline(best_depth, color="gray", ls=":", label=f"Best depth={best_depth}")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree: Depth vs Accuracy", fontweight="bold")
plt.legend()
plt.show()

<a id = "4"></a>
<center><h1 style = "background:#c1440e ;color:white;border:0;font-weight:bold">Visualise the Optimal Tree</h1></center>

In [ ]:
best_dt = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_dt.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(20, 8), dpi=80)
plot_tree(best_dt, feature_names=X.columns.tolist(),
          class_names=data.target_names, filled=True, rounded=True,
          fontsize=8, ax=ax)
plt.title(f"Optimal Decision Tree (max_depth={best_depth})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

<a id = "5"></a>
<center><h1 style = "background:#3d1700 ;color:white;border:0;font-weight:bold">Evaluation of Model</h1></center>

In [ ]:
y_pred = best_dt.predict(X_test)
acc  = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=data.target_names))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4), dpi=100)
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Feature importance
perm = permutation_importance(best_dt, X_test, y_test, n_repeats=20, random_state=42)
top_n = 12
idx = np.argsort(perm.importances_mean)[::-1][:top_n]

plt.figure(figsize=(9, 5), dpi=100)
plt.barh([X.columns[i] for i in idx[::-1]],
         perm.importances_mean[idx[::-1]],
         xerr=perm.importances_std[idx[::-1]],
         color="#c1440e", edgecolor="white")
plt.xlabel("Permutation Importance")
plt.title("Top Feature Importances", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
results = [accuracy_score(y_test, y_pred),
           __import__("sklearn.metrics", fromlist=["precision_score"]).precision_score(y_test, y_pred),
           __import__("sklearn.metrics", fromlist=["recall_score"]).recall_score(y_test, y_pred),
           __import__("sklearn.metrics", fromlist=["f1_score"]).f1_score(y_test, y_pred)]
pd.DataFrame({"Metric": ["Accuracy", "Precision", "Recall", "F1"],
              "Score": results}).round(4)